In [ ]:
#| default_exp core

In [ ]:
from dialoghelper import *
import inspect
from safepyrun import find_var

# dialoghelper

> Inspect and edit Solveit dialogs, run messages, manage kernels, and communicate with browser pages

Use these helpers from a Solveit kernel. Most examples below need a running Solveit server and a current dialog. They change that dialog and include examples that run prompts or restart and delete other dialogs. Use a disposable dialog to try them.

In [ ]:
#| export
import os,asyncio,json

from typing import Dict
from os.path import normpath
from fastcore.xml import *
from fastcore.xml import Msgs
from fastcore.meta import splice_sig, delegates

from fastcore.utils import *
from fastcore.xtras import str_diff
from inspect import currentframe
import httpx2
from comm import create_comm
from IPython.display import Markdown,HTML
from urllib.parse import urlencode, quote
from jupyasyncclient import CommTransport, http_client, request_json
from safepyrun import load_ipython_extension
from fastcore.tools import *
from fastcore.nbio import item2xml

In [ ]:
from fastcore import tools
from fastcore.test import *

## Basics

In [ ]:
a = 1
find_var('a')

1

In [ ]:
#| export
dh_settings = dict(kid=os.environ.get('RUSTYGATE_KERNEL_ID'), path=os.environ.get('RUSTYGATE_KERNEL_PATH'))

In [ ]:
#| export
try: load_ipython_extension(get_ipython())
except NameError: pass

In [ ]:
names_containing('ipython')

['get_ipython', 'load_ipython_extension', 'ipython_shell', 'in_ipython']

## Reaching rustygate

dialoghelper calls rustygate's API through a `rustygate` comm in a gateway kernel, without a URL or token. Outside one, it sends HTTP to `RUSTYGATE_URL`. `jupyasyncclient.http` owns the transport and JSON request mechanics; dialoghelper supplies its gateway configuration and sends the same method, route, query parameters and body over either transport.


In [ ]:
#| export
def _http_options():
    if dh_settings.get('kid'): return dict(base_url='http://rustygate', transport=CommTransport(), timeout=None)
    return dict(base_url=dh_settings.get('rusty') or os.environ['RUSTYGATE_URL'], timeout=None)

async def _rg(method, route, json=None, **params):
    return await request_json(method, route, json=json, params=params, **_http_options())

async def refresh_dlg():
    "Ask every browser showing this dialog to render its current messages and templates"
    await _rg('POST', f"/api/kernels/{dh_settings['kid']}/refresh")

Here the comm carries a request for this kernel's model. Its `path` is the notebook this code runs in.

After filling templates programmatically, call `await refresh_dlg()` to update every browser showing this dialog. It requests a full render without reconnecting or restarting the kernel; it does not wait for browsers to finish rendering.

In [ ]:
test_eq((await _rg('GET', f"/api/kernels/{dh_settings['kid']}"))['path'], dh_settings['path'])

In [ ]:
#| export
def find_dname(dname=None, required=True):
    "Return `dname` as a gateway-root-relative notebook name without a leading slash, adding a missing `.ipynb`. A leading `/` starts from the gateway root, other names are relative to the current dialog's folder, and a trailing `/` names a folder."
    if dname and not dname.endswith(('.ipynb','/')): dname += '.ipynb'
    if dname and dname.startswith('/'): return dname.strip('/')
    curr = dh_settings.get('path')
    if not curr:
        if not required: return None
        raise ValueError("No dialog context: Please pass absolute `dname` starting with '/'")
    curr = curr.lstrip('/')
    if not dname: return curr
    res = normpath(Path(curr).parent/dname)
    assert '../' not in res, "Path traversal not permitted"
    return res

In [ ]:
find_dname()

'nbs/00_core.ipynb'

In [ ]:
test_eq(find_dname('index'), find_dname('index.ipynb'))
find_dname('index')

'nbs/index.ipynb'

In [ ]:
find_dname('../index.ipynb')

'index.ipynb'

In [ ]:
find_dname('/foo/bar.ipynb')

'foo/bar.ipynb'

`browser` runs JavaScript in each browser showing this dialog. It opens a `browser` comm, and Solveit runs the code in every connected browser as the body of an async function. That function receives `done`, which answers the call, and `rid`, the call's id. Code that answers later from an event handler can instead pass `rid` on and call `pushData(rid, {result})`, as the solveit-chrome extension does.

The first browser to answer supplies the result. A JavaScript error raises `RuntimeError`, and no answer within `timeout` seconds raises `TimeoutError`. With `wait=False`, `browser` returns at once. The code never appears in a cell's output, so reloading a page doesn't run it again.

In [ ]:
#| export
def _browser_comm(js): return create_comm(target_name='browser', data=dict(code=js))

async def browser(
    js:str, # Body of an async JavaScript function, which receives `done` and `rid`
    wait:bool=True, # Wait for the first browser to call `done`?
    timeout:float=15 # Seconds to wait for an answer
):
    "Run `js` in each browser showing this dialog; with `wait`, return the value the first browser passes to `done`"
    c = _browser_comm(js)
    if not wait: return c.close()
    q,loop = asyncio.Queue(),asyncio.get_running_loop()
    c.on_msg(lambda m: loop.call_soon_threadsafe(q.put_nowait, m['content']['data']))
    try: res = await asyncio.wait_for(q.get(), timeout)
    except TimeoutError: raise TimeoutError(f'No browser answered within {timeout}s') from None
    finally: c.close()
    if 'error' in res: raise RuntimeError(res['error'])
    return res.get('result')

def add_html(
    content:str, # HTML whose `hx-swap-oob` elements are swapped into the page
):
    "Swap `content`'s out-of-band elements into each browser showing this dialog"
    _browser_comm(f"htmx.swap(document.body, {json.dumps(to_xml(content))}, {{swapStyle: 'none'}})").close()

In [ ]:
#| eval: false
await browser("done({width: window.innerWidth})")

In [ ]:
#| eval: false
await browser("done(document.querySelectorAll('.msg').length)")

`add_html` swaps the `hx-swap-oob` elements of some HTML into each browser's page. It runs `htmx.swap` through `browser`, and returns without waiting:

In [ ]:
add_html(Div(P('Hi'), hx_swap_oob='beforeend:#dialog-container'))

{'success': 'Content added to DOM'}

`realpath()` returns the gateway's on-disk root. `realpath('')` returns the current notebook's folder. Other relative paths use that folder, regardless of the kernel's cwd. A leading slash uses the gateway root. Targets need not exist. `..` can move up within the root, and a path that resolves outside it raises `ValueError`. `realpath` makes no gateway call and doesn't need awaiting, so a `$` reference can use it.

These helpers resolve names against `dh_settings['path']`, the kernel's notebook, which follows renames. An unbound kernel can resolve root-qualified paths but has no notebook-relative base. `list_dialogs()` lists the current notebook's folder by default and keeps the existing notebook-extension and folder-suffix conventions.

In [ ]:
#| export
def realpath(
    subpath:str='/' # Gateway-root-relative with a leading slash, otherwise relative to the current notebook's folder
) -> str:
    "The on-disk path of `subpath`, which must stay within the gateway root"
    root = data_root().resolve()
    p = (root/subpath.lstrip('/') if subpath.startswith('/') else root/Path(find_dname()).parent/subpath).resolve()
    if not p.is_relative_to(root): raise ValueError(f'{subpath!r} is outside the gateway root')
    return str(p)

In [ ]:
realpath('data/results')

'/Users/jhoward/aai-ws/dialoghelper/nbs/data/results'

In [ ]:
#| export
async def list_dialogs(
    subpath:str='', # Path under data root (absolute with `/`, else relative to current dialog's folder)
    depth:int=1 # Directory depth
) -> dict:
    "List dialogs and folders under `subpath`. Folders have `/` suffix."
    return await _rg('GET', '/api/dialogs', **_target(subpath), depth=depth)

In [ ]:
(await list_dialogs())['items'][-4:]

['06_utils.ipynb', '07_test.ipynb', 'data/', 'index.ipynb']

In [ ]:
(await list_dialogs('/'))['items'][:5]

['SOLVEIT.ipynb', '_proc/', 'data/', 'dialoghelper/', 'dialoghelper.egg-info/']

## Messages API

In [ ]:
#| export
def data_root():
    "The gateway root recorded at startup, or derived from the current notebook's folder"
    if r := dh_settings.get('root'): return Path(r)
    if not (nm := find_dname(required=False)): raise RuntimeError('data_root requires a bound gateway kernel')
    root = Path.cwd()
    for _ in Path(nm).parent.parts: root = root.parent
    return root

The reading and editing tools use Rustygate's messages API. Rustygate resolves dialog paths, selects messages, converts between message and notebook formats, and applies changes to the current notebook state.

Text edits read content before sending a replacement. Writes do not check whether that content changed meanwhile; the last write to a field wins. Solveit updates its view from the gateway's broadcasts.

The kernel's bootstrap records the gateway root in `dh_settings['root']`. `data_root` returns it for helpers that use local filesystem paths.

In [ ]:
#| export
def _target(dname=''):
    return dict(kid=dh_settings.get('kid'), path=dname or None)

## View/edit dialog

In [ ]:
#| export
def _lnhashs_content(s, start_line=1, end_line=None):
    "Render `s` as `lineno|hash|` prefixed lines, sliced by optional 1-based line range (`end_line` None or -1 for EOF)"
    lines = [lnhash(i,l)+l for i,l in enumerate(s.splitlines(), 1)]
    return '\n'.join(lines[start_line-1:len(lines) if end_line in (None,-1) else end_line])

In [ ]:
s = 'x = 21\nx*2\ndone'
test_eq(_lnhashs_content(s), '\n'.join(lnhash(i,l)+l for i,l in enumerate(s.splitlines(),1)))
test_eq(_lnhashs_content(s, 2, -1).splitlines()[0], lnhash(2,'x*2')+'x*2')
test_eq(len(_lnhashs_content(s, 1, 2).splitlines()), 2)

In [ ]:
#| export
def _mid(id):
    "Message `id` without the `_` prefix of its DOM form, as in `_f5fd1d54`"
    return id.removeprefix('_') if id else id

def _ids(ids):
    "Message ids from a comma-separated string or a list, with blank entries dropped"
    if isinstance(ids, str): ids = ids.split(',')
    return [o.strip() for o in listify(ids) if o.strip()]

async def _caller():
    "Id of the cell this code runs for: the running cell, or a tool call's prompt cell (None outside any cell)"
    if not (kid := dh_settings.get('kid')): return None
    return (await _rg('GET', f'/api/kernels/{kid}/caller'))['cell']

In [ ]:
#| export
async def read_msg(
    n:int=-1,      # Message index (if relative, +ve is downwards)
    relative:bool=True,  # Is `n` relative to current message (True) or absolute (False)?
    id:str=None,  # Message id to find (defaults to current message)
    start_line:int=1, # Starting line to view
    end_line:int=None, # End line (defaults to last line if None; -1 for EOF)
    nums:bool=False, # Whether to show line numbers
    lnhashs:bool=False, # Show exhash `lineno|hash|` addresses instead of line numbers?
    dname:str='' # Dialog to get info for; defaults to current dialog
    ) -> dict|None:
    """Get the message indexed in the current dialog, or None when there's no such message.
    NB: Messages in the current dialog above the current message are *already* visible; use this only when you need line numbers for editing operations, or for messages not in the current dialog or below the current message.
    - To get the exact message use `n=0` and `relative=True` together with `id`.
    - To get a relative message use `n` (relative position index).
    - To get the nth message use `n` with `relative=False`, e.g `n=0` first message, `n=-1` last message."""
    if start_line<1: raise ValueError('start_line must be at least 1')
    found = await _rg('GET', '/api/messages', **_target(dname), id=id, n=n, relative=relative)
    if not found: return None
    res = AttrDict(found[0])
    if lnhashs: res['content'] = _lnhashs_content(res['content'], start_line, end_line)
    elif start_line!=1 or end_line is not None:
        lines = res['content'].splitlines()
        e = len(lines) if end_line in (None,-1) else end_line
        lines = lines[start_line-1:e]
        if nums: lines = [f'{i+start_line:6d} │ {l}' for i,l in enumerate(lines)]
        res['content'] = '\n'.join(lines)
    elif nums: res['content'] = '\n'.join(f'{i:6d} │ {l}' for i,l in enumerate(res['content'].splitlines(), 1))
    return res

In [ ]:
#| export
def _trunc_middle(s, limit=500, msg='\n…TRUNCATED…\n'):
    "Middle-out truncation, keeping both ends of `s`"
    if len(s) <= limit: return s
    i = limit//2
    return s[:i] + msg + s[len(s)-(limit-i):]

def _msgs2xml(ds, ids=True):
    "Message dicts as concise XML, in the shared `item2xml` grammar"
    its = [item2xml({'note':'markdown'}.get(d['msg_type'], d['msg_type']), d.get('content',''), d.get('output') or '',
        id=d.get('id') if ids else None, meta=d.get('meta')) for d in ds]
    return to_xml(Msgs(*its), do_escape=False)

In [ ]:
#| export
async def find_msgs(
    re_pattern:str='', # Optional regex to search for (re.DOTALL+re.MULTILINE is used)
    msg_type:str=None, # optional limit by message type ('code', 'note', or 'prompt')
    before:int=0,  # Include additional n msgs before matches
    after:int=0,   # Include additional n msgs after matches
    context:int=None, # Include additional n msgs around matches (default 1, or 0 when `headers_only`)
    use_case:bool=False, # Use case-sensitive matching?
    use_regex:bool=True, # Use regex matching?
    only_err:bool=None, # Only messages with errors if True, only those without if False, all if None
    only_exp:bool=None, # Only exported messages if True, only unexported if False, all if None
    only_chg:bool=False, # Only messages whose input differs from Git HEAD
    ids:str='', # Optionally filter by comma-separated list of message ids
    limit:int=None, # Optionally limit number of returned items
    include_output:bool=True, # Include output in returned dict?
    include_meta:bool=True, # Include all additional message metadata
    as_xml:bool=False, # Use concise unescaped XML output format
    nums:bool=False, # Show line numbers?
    trunc_out:bool=False, # Middle-out truncate code output to 100 characters?
    trunc_in:bool=False, # Middle-out truncate cell content to 80 characters?
    headers_only:bool=False, # Only return note messages that are headers (first line only); cannot be used together with `header_section`
    header_section:str=None, # Find section starting with this header; returns it plus all children
    include_skipped:bool=False, # Include messages hidden from AI (skipped)?
    dname:str='' # Dialog to get info for; defaults to current dialog
)->list[dict]: # Messages in requested dialog that contain the given information
    """Often it is more efficient to call `view_dlg` (not `find_msgs`) to see the whole dialog, so you can use it all from then on.
    Message ids are identical to those in LLM chat history, so do NOT call this to view a specific message if it's in the chat history--instead use `view_msg`.
    Do NOT use find_msgs to view message content in the current dialog above the current prompt -- these are *already* provided in LLM context, so just read the content there directly. (NB: LLM context only includes messages *above* the current prompt, whereas `find_msgs` can access *all* messages.)
    To refer to a found message from code, use its `id` field."""
    found = await _rg('GET', '/api/messages', **_target(dname), re_pattern=re_pattern, msg_type=msg_type,
        before=before, after=after, context=context, use_case=use_case, use_regex=use_regex, only_err=only_err, only_exp=only_exp,
        only_chg=only_chg, ids=','.join(_ids(ids)) if ids else None, limit=limit, headers_only=headers_only,
        header_section=header_section, include_skipped=include_skipped, raw_prompt_reply=as_xml)
    res = []
    for o in found:
        flds = {'id','content','msg_type','exported'}
        if include_output or o['msg_type']=='prompt': flds.add('output')
        if include_meta: flds |= {'meta','skipped','pinned','bookmark','i_collapsed','o_collapsed','heading_collapsed'}
        o = AttrDict({k:v for k,v in o.items() if k in flds})
        if nums: o['content'] = '\n'.join(f'{i+1:6d} │ {l}' for i,l in enumerate(o['content'].splitlines()))
        if trunc_in: o['content'] = _trunc_middle(o['content'], 160, msg='\n…\n')
        if trunc_out and isinstance(o.get('output'), str) and o['msg_type']!='prompt': o['output'] = _trunc_middle(o['output'], 100, msg='\n…\n')
        if headers_only: o['content'] = o['content'].split('\n', 1)[0] + ('\n<TRUNCATED>' if '\n' in o['content'] else '')
        res.append(o)
    return _msgs2xml(res) if as_xml else res

In [ ]:
1+1

2

In [ ]:
r = await find_msgs(r'1\+1', include_meta=False, include_output=True, context=0)
r

[{'id': '8ce548d6',
  'content': '1+1',
  'output': '2',
  'msg_type': 'code',
  'exported': False}]

In [ ]:
hl_md(await find_msgs(r'1\+1', include_meta=False, as_xml=True, context=0))

```html
<msgs><code id="8ce548d6">1+1<out>2</out></code></msgs>
```

In [ ]:
#| export
async def view_dlg(
    dname:str='', # Dialog to get info for; defaults to current dialog
    msg_type:str=None, # optional limit by message type ('code', 'note', or 'prompt')
    nums:bool=False, # Whether to show line numbers
    include_output:bool=False, # Include output in returned dict?
    trunc_out:bool=True, # Middle-out truncate code output to 100 characters (only applies if `include_output`)?
    trunc_in:bool=False, # Middle-out truncate cell content to 80 characters?
    include_skipped:bool=False, # Include messages hidden from AI (skipped)?
) -> str:
    "Concise XML view of all messages (optionally filtered by type), not including metadata. Often it is more efficient to call this to see the whole dialog at once (including line numbers if needed), instead of running `find_msgs` or `view_msg` multiple times."
    return await find_msgs(msg_type=msg_type, dname=dname, as_xml=True, nums=nums,
        include_meta=False, include_output=include_output, trunc_out=trunc_out, trunc_in=trunc_in, include_skipped=include_skipped)

In [ ]:
hl_md((await view_dlg(nums=True))[:500])

```html
<msgs><code id="955b9784">     1 │ #| default_exp core</code><code id="a982e24d">     1 │ from dialoghelper import *</code><markdown id="0aafe008">     1 │ # dialoghelper
     2 │ 
     3 │ > Inspect and edit Solveit dialogs, run messages, manage kernels, and communicate with browser pages
     4 │ 
     5 │ Use these helpers from a Solveit kernel. Most examples below need a running Solveit server and a current dialog. They change that dialog and include examples that run prompts or restart and 
```

In [ ]:
#| export
async def add_msgs(
    msgs:list[dict], # Message dictionaries with content and the same per-message options as add_msg
    msgid:str=None, # Message to place the batch beside; defaults to the calling cell in its own dialog
    position:str='after', # 'after' or 'before' msgid, or the dialog's 'start' or 'end'
    dname:str='' # Dialog to add to; defaults to current dialog
) -> list[str]:
    "Add messages in their input order with one request, returning their ids"
    return (await _rg('POST', '/api/messages', **_target(dname),
        json=dict(op='add', msgs=msgs, msgid=msgid, position=position)))['added_ids']
async def add_msg(
    content:str, # Content of the message (i.e the message prompt, code, or note text)
    msg_type:str='note', # Message type, can be 'code', 'note', or 'prompt'
    msgid:str=None, # Message to place the new one beside; default: the calling cell in its own dialog, else the end (or start for `position='before'`)
    position:str='after', # 'after' or 'before' `msgid`, or the dialog's 'start' or 'end'
    dname:str='', # Dialog to add to; defaults to current dialog
    output:str='', # Prompt/code output; Code outputs must be .ipynb-compatible JSON array
    exported:int=0, # Mark message as exported (stored as nbdev `export` metadata)?
    skipped:int=0, # Hide message from prompt?
    i_collapsed:int=0, # Collapse input?
    o_collapsed:int=0, # Collapse output?
    heading_collapsed:int=0, # Collapse heading section?
    pinned:int=0, # Pin to context?
    bookmark:int=0, # Bookmark number, 1 to 9, or 0 for none
)->str: # Message ID of newly created message
    "Add a message to a dialog, returning its id"
    msg = dict(content=content, msg_type=msg_type, output=output, exported=exported, skipped=skipped, i_collapsed=i_collapsed,
        o_collapsed=o_collapsed, heading_collapsed=heading_collapsed, pinned=pinned, bookmark=bookmark)
    return (await add_msgs([msg], msgid=msgid, position=position, dname=dname))[0]

Without `msgid`, `add_msg` places the new message after the calling cell: the cell whose code is running, or the prompt whose tool call is running. `position='before'` places it before that cell instead. Successive calls therefore come out in reverse order. Use `add_msgs` to add a list of message dictionaries in input order with one request; each dictionary takes `content` and the same per-message options as `add_msg`. When there's no calling cell, or the message goes to another dialog, it goes at the end, or at the start for `position='before'`. `position='start'` and `position='end'` put it at the dialog's start or end, whatever the calling cell. The id of the prompt a model is answering is the `sid` attribute of its `<prompt>` tag.

In [ ]:
_id = await add_msg('testing')
_id

'9ed1250c'

In [ ]:
#| export
async def read_msgid(
    id:str,  # Message id to find
    start_line:int=1, # Starting line to view
    end_line:int=None, # End line (defaults to last line if None; -1 for EOF)
    nums:bool=False, # Whether to show line numbers
    lnhashs:bool=False, # Show exhash `lineno|hash|` addresses instead of line numbers?
    dname:str='', # Dialog to get message from; defaults to current dialog
    add_to_dlg:bool=False # Whether to add message content to current dialog (as a raw message)
) -> dict|None:
    """Get message `id`, or None when there's no such message. Message IDs can be view directly in LLM chat history/context, or found in `find_msgs` results.
    Use `add_to_dlg` if the LLM or human may need to refer to the message content again later."""
    res = await read_msg(0, id=id, start_line=start_line, end_line=end_line, nums=nums, lnhashs=lnhashs, dname=dname)
    if add_to_dlg and res: await add_msg(res['content'], msg_type='raw')
    return res

In [ ]:
r = await read_msg(-2)
print((await read_msg(-2)).content)

_id = await add_msg('testing')
_id


`read_msg` returns a dictionary with attribute access. Access its fields as dictionary keys, such as `r['content']`, or attributes, such as `r.content`. Functions that take message ids also accept the DOM form that Solveit's page and message links use, with a leading `_`, such as `_f5fd1d54`.

In [ ]:
bmsg = await add_msg('just below the calling cell')
emsg = await add_msg('at the end', position='end')

In [ ]:
test_eq((await read_msg(-1))['id'], bmsg)
test_eq((await read_msg(-1, relative=False))['id'], emsg)
test_eq((await read_msgid('_'+bmsg))['id'], bmsg)

In [ ]:
#| export
async def view_msg(
    id:str,  # Message id to view
    dname:str='', # Dialog to get message from; defaults to current dialog
    nums:bool=True, # Whether to show line numbers
    lnhashs:bool=False, # Show exhash `lineno|hash|` addresses instead of line numbers?
    start_line:int=1, # Starting line to view. Rarely needed--read whole message in nearly all cases instead
    end_line:int=None, # End line (defaults to last line if None; -1 for EOF)
    incl_out:bool=False, # Append the message's output in an `<out>` block?
    trunc_out:bool=True, # Truncate an included output to ~512 chars?
    add_to_dlg:bool=False # Whether to add message content to current dialog (as a raw message)
) -> str:
    """Views the *content* of message `id`. Same as `read_msgid(...)['content']`, defaulting to `nums=True`.
    Use `add_to_dlg` if the LLM or human may need to refer to the message content again later."""
    r = await read_msg(0, id=id, start_line=start_line, end_line=end_line, nums=nums, lnhashs=lnhashs, dname=dname)
    if not r: return 'No such message.'
    res = r['content']
    if incl_out and (o := r.get('output')): res += f"\n<out>\n{truncstr(o, 512) if trunc_out else o}\n</out>"
    if add_to_dlg: await add_msg(res, msg_type='raw')
    return res

In [ ]:
print((await view_msg(r.id)))

     1 │ _id = await add_msg('testing')
     2 │ _id


In [ ]:
#| export
def msg_ref(id, dname=None):
    "Markdown ref to a message — same-dialog `#_id` or cross-dialog `#dname/_id` (anchors target DOM ids, which carry a `_` prefix)"
    id = '_'+_mid(id)
    if not dname: return f'#{id}'
    return f'#{find_dname(dname)}/{id}'

In [ ]:
#| export
async def _apply_logged(command, logs, dname=''):
    if command: await _rg('POST', '/api/messages', **_target(dname), json=command)
    if logs: await add_msg('\n\n'.join(logs))

def _change_log(id, diff, dname):
    return f'> Updated {msg_ref(id, dname)}\n\n' + (f'```diff\n{diff}\n```' if diff else 'No changes.')

async def del_msgs(
    ids:str=None, # Comma-separated ids of message(s) to delete
    dname:str='', # Dialog to get info for; defaults to current dialog
    log_changed:bool=False # Add a note showing the deleted content?
) -> list:
    "Delete exactly the named messages (a collapsed heading's hidden section stays). DO NOT USE THIS unless you have been explicitly instructed to delete messages."
    ids = _ids(ids)
    if not ids: return []
    found = (await _rg('GET', '/api/messages', **_target(dname), ids=','.join(ids), context=0,
        include_skipped=True)) if log_changed else []
    logs = [f"> Deleted {msg_ref(m['id'], dname)}\n\n```\n{m['content']}\n```" for m in found]
    await _apply_logged(dict(op='delete', ids=ids), logs, dname)
    return ids

In [ ]:
await del_msgs(f'{bmsg},{emsg}')
await del_msgs(_id)

['9ed1250c']

In [ ]:
#| export
def _umsg(
    content:str|None=None, # Content of the message (i.e the message prompt, code, or note text)
    msg_type: str|None = None, # Message type, can be 'code', 'note', or 'prompt'
    output:str|None = None, # Prompt/code output; Code outputs must be .ipynb-compatible JSON array
    exported: int | None = None, # Set export state (nbdev `export` metadata)?
    skipped: int | None = None, # Hide message from prompt?
    i_collapsed: int | None = None, # Collapse input?
    o_collapsed: int | None = None, # Collapse output?
    heading_collapsed: int | None = None, # Collapse heading section?
    pinned: int | None = None, # Pin to context?
    meta: dict | None = None, # Replace message meta wholesale
    mergemeta: dict | None = None # Deep-merge into message meta; a `None` value deletes its key
): ...

In [ ]:
#| export
@delegates(_umsg)
async def update_msg(
    id:str=None, # id of message to update (if None, uses current message)
    msg:Optional[Dict]=None, # Dictionary of field keys/values to update
    dname:str='', # Dialog to get info for; defaults to current dialog
    log_changed:bool=False, # Add a note showing the diff?
    **kwargs) -> str:
    """Update an existing message. Provide either `msg` OR field key/values to update.
    - Use `content` param to update contents.
    - Only include parameters to update--missing ones will be left unchanged."""
    if msg: kwargs |= msg.get('msg', msg)
    supplied_id = kwargs.pop('id', None)
    id = id or supplied_id or await _caller()
    old = await read_msg(n=0, id=id, dname=dname) if log_changed else None
    await _rg('POST', '/api/messages', **_target(dname), json=dict(op='update', msgs=[dict(id=id, msg=kwargs)]))
    if log_changed:
        new = kwargs.get('content')
        await add_msg(_change_log(id, str_diff(old['content'], old['content'] if new is None else new), dname))
    return id

In [ ]:
_id = await add_msg('testing')

In [ ]:
_id = await update_msg(_id, content='toasting')

In [ ]:
_id = await update_msg(_id, skipped=1)

`update_msg` sends only the fields you pass. `meta=` replaces the metadata dictionary. `mergemeta=` recursively merges dictionaries and deletes keys whose value is `None`. Rustygate applies both against the message's current state.

`log_changed=True` appends an audit note in the calling dialog, including when the edited messages belong to another dialog.

In [ ]:
await update_msg(_id, mergemeta=dict(pins=dict(a=1)))
test_eq((await read_msgid(_id))['meta']['pins'], {'a': 1})
await update_msg(_id, mergemeta=dict(pins=None))
assert 'pins' not in (await read_msgid(_id))['meta']

In [ ]:
msg = await read_msgid(_id)
msg['content'] = 'toasted'
await update_msg(msg=msg)

'e49c9729'

In [ ]:
await del_msgs(_id)

['e49c9729']

In [ ]:
_edit_id = await add_msg('This message should be found.\n\nThis is a multiline message.')
_edit_id

'c99d7c11'

In [ ]:
print((await read_msg())['content'])

This message should be found.

This is a multiline message.


In [ ]:
print((await read_msg(n=0, id=_edit_id, nums=True))['content'])

     1 │ This message should be found.
     2 │ 
     3 │ This is a multiline message.


In [ ]:
print((await read_msg(n=0, id=_edit_id, nums=True, start_line=2, end_line=3))['content'])

     2 │ 
     3 │ This is a multiline message.


In [ ]:
codeid = (await read_msg())['id']

In [ ]:
#| export
async def copy_msgs(
    ids:str, # Comma-separated ids of message(s) to copy
    cut:bool=False, # Cut message(s)? (If not, copies)
    dname:str='' # Dialog to copy messages from; defaults to current dialog
) -> list:
    "Copy messages to the gateway's clipboard, replacing its contents, and delete them if `cut`"
    ids = _ids(ids)
    await _rg('POST', '/api/messages', **_target(dname), json=dict(op='copy', ids=ids, cut=cut))
    return ids

In [ ]:
#| export
async def paste_msgs(
    msgid:str=None, # Message to paste beside; default: the calling cell in its own dialog, else the end (or start for `position='before'`)
    position:str='after', # 'after' or 'before' `msgid`, or the dialog's 'start' or 'end'
    dname:str='' # Dialog to paste into; defaults to current dialog
) -> list:
    "Paste the gateway clipboard's messages where `add_msg` would place a new one, returning their ids"
    return (await _rg('POST', '/api/messages', **_target(dname),
        json=dict(op='paste', msgid=msgid, position=position)))['added_ids']

`paste_msgs` places the pasted messages where `add_msg` would place a new one. These functions use the gateway's clipboard, which Solveit's copy and paste use too, so messages copied in the browser can be pasted from code and the other way round. A copy replaces the clipboard's contents. The first paste after a cut keeps the cut messages' ids, so links to them keep working. Every other paste gives the copies new ids.

In [ ]:
await copy_msgs(codeid)

['eda5f04b']

In [ ]:
with expect_fail(httpx2.HTTPStatusError): await copy_msgs('nosuchid')

In [ ]:
tgt = (await read_msg())['id']

In [ ]:
await paste_msgs(tgt)

['bdaa7a50']

In [ ]:
newmsg = await read_msg(1, id=tgt)
test_eq(newmsg['content'], (await read_msgid(codeid))['content'])
newmsg['content']

"print((await read_msg(n=0, id=_edit_id, nums=True, start_line=2, end_line=3))['content'])"

In [ ]:
await del_msgs(newmsg['id'])

['bdaa7a50']

In [ ]:
#| export
mermaid_url = "https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs"

def enable_mermaid():
    return Script(Safe("""
if (window.mermaid) mermaid.run()
else {
    import('%s').then(m => {
        window.mermaid = m.default;
        window.mermaid.run();
        htmx.onLoad(elt => {
            if (elt.matches('div.mermaid, pre.mermaid') || htmx.findAll(elt, 'div.mermaid, pre.mermaid')) window.mermaid.run();
        });
    });
}""" % mermaid_url), type="module")

In [ ]:
mermaid_script = enable_mermaid()
assert 'm => {' in to_xml(mermaid_script)
mermaid_script

<script type="module">
if (window.mermaid) mermaid.run()
else {
    import('https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs').then(m => {
        window.mermaid = m.default;
        window.mermaid.run();
        htmx.onLoad(elt => {
            if (elt.matches('div.mermaid, pre.mermaid') || htmx.findAll(elt, 'div.mermaid, pre.mermaid')) window.mermaid.run();
        });
    });
}</script>

In [ ]:
#| export
def mermaid(code, cls="mermaid", **kwargs):
    "A mermaid diagram"
    return Div(code, cls=cls, **kwargs)

In [ ]:
mermaid('graph LR; A[Start] --> B[Process]; B --> C[End];')

<div class="mermaid">graph LR; A[Start] --&gt; B[Process]; B --&gt; C[End];</div>

You can also add to a note:

    ```mermaid
    graph LR
    A[Start] --> B[Process]
    B --> C[End]
    ```

This renders as: 

```mermaid
graph LR
A[Start] --> B[Process]
B --> C[End]
```

In [ ]:
#| export
async def toggle_header(
    id:str, # id of markdown header note message to toggle collapsed state
    dname:str='' # Dialog to toggle in; defaults to current dialog
) -> dict:
    "Toggle collapsed header state for `id`"
    await _rg('POST', '/api/messages', **_target(dname), json=dict(op='toggle', ids=[id], key=['heading_collapsed']))
    return {'success': True, 'id': id}

In [ ]:
#| export
async def toggle_bookmark(
    id:str, # id of message to toggle bookmark on
    n:int, # Bookmark number (1-9)
    dname:str='' # Dialog to set bookmark in; defaults to current dialog
) -> dict:
    "Toggle numbered bookmark (1-9) on a message, clearing it from any other message when setting"
    await _rg('POST', '/api/messages', **_target(dname), json=dict(op='bookmark', id=id, n=n))
    return {'success': True, 'id': id}

In [ ]:
#| export
async def toggle_export(
    ids:str|list, # Message id(s) to toggle (comma-separated str, or list)
    dname:str='' # Dialog to get info for; defaults to current dialog
) -> dict:
    "Toggle the nbdev `export` directive (meta form) on each message, independently"
    await _rg('POST', '/api/messages', **_target(dname),
        json=dict(op='toggle', ids=_ids(ids), key=['nbdev','export'], value='true'))
    return {'success': True}

In [ ]:
#| export
def _toggle_comment(s):
    "Comment every line when any is uncommented, else uncomment all"
    lines = s.splitlines()
    if any(l.strip() and not l.lstrip().startswith('#') for l in lines): return '\n'.join('# '+l for l in lines)
    return '\n'.join(l[2:] if l.startswith('# ') else l[1:] if l.startswith('#') else l for l in lines)

async def toggle_comment(
    id:str, # id of code message (or comma-separated ids) to toggle comments on
    dname:str='' # Dialog to toggle comments in; defaults to current dialog
) -> dict:
    "Toggle line comments on code message(s). If any lines are uncommented, comments all; otherwise uncomments all."
    found = await _rg('GET', '/api/messages', **_target(dname), ids=','.join(_ids(id)), context=0,
        include_skipped=True)
    msgs = [dict(id=m['id'], msg=dict(content=_toggle_comment(m['content']))) for m in found if m['msg_type']=='code']
    if msgs: await _rg('POST', '/api/messages', **_target(dname), json=dict(op='update', msgs=msgs))
    return {'success': True}

In [ ]:
await toggle_comment(codeid) # comment
await toggle_comment(codeid) # uncomment

{'success': True}

In [ ]:
with expect_fail(httpx2.HTTPStatusError): await toggle_comment('_fake', dname='/dlg/nonexistent.ipynb')

#### test header

#### header end

In [ ]:
hdid = (await read_msg())['id']

In [ ]:
await toggle_header(hdid)

{'success': True, 'id': '9bba132a'}

In [ ]:
with expect_fail(httpx2.HTTPStatusError): await toggle_header('_fake', dname='/dlg/nonexistent.ipynb')

### Dlg conveniences

These functions manage dialogs and their kernels through rustygate. `create_or_run_dialog` gets or starts the kernel bound to a dialog, creating the dialog if it's missing. A new dialog starts with its folder's `TEMPLATE.ipynb` cells and settings. When the templates set no mode, it takes the current dialog's mode. `restart_dialog` keeps the kernel's id, and `rm_dialog` stops the kernel before deleting the file.

In [ ]:
#| export
async def _kernel_for(name):
    "The kernel bound to the gateway-root-relative dialog `name`, or None"
    return first(await _rg('GET', '/api/kernels', path=name))

async def create_or_run_dialog(
    name:str, # Name/path of the dialog (relative to current dialog's folder, or absolute if starts with '/')
)->dict: # The dialog's kernel model
    "Create a new dialog, or start an existing one's kernel. A new dialog takes its mode from the current one."
    view = await _rg('GET', f"/api/kernels/{dh_settings['kid']}/cells", fields='meta', limit=0) if dh_settings.get('kid') else {}
    mode = view.get('metadata', {}).get('solveit', {}).get('mode')
    return await _rg('POST', '/api/kernels', json=dict(path=find_dname(name), dialog=dict(mode=mode) if mode else {}))

In [ ]:
tk = await create_or_run_dialog('test_dialog.ipynb')
tk['path']

'nbs/test_dialog.ipynb'

In [ ]:
#| export
async def restart_dialog(
    name:str, # Name/path of the dialog (relative to current dialog's folder, or absolute if starts with '/')
)->dict: # The dialog's kernel model
    "Restart a dialog's kernel, starting the dialog first if needed"
    k = await _kernel_for(find_dname(name))
    return await _rg('POST', f"/api/kernels/{k['id']}/restart") if k else await create_or_run_dialog(name)

In [ ]:
test_eq((await restart_dialog('test_dialog.ipynb'))['id'], tk['id'])

In [ ]:
#| export
async def stop_dialog(
    name:str, # Name/path of the dialog (relative to current dialog's folder, or absolute if starts with '/')
):
    "Stop a dialog's kernel, if it's running"
    if k := await _kernel_for(find_dname(name)): await _rg('DELETE', f"/api/kernels/{k['id']}")

In [ ]:
await stop_dialog('test_dialog.ipynb')
assert not await _kernel_for(tk['path'])

In [ ]:
#| export
async def rm_dialog(
    name:str, # Name/path of the dialog, or of an empty folder, to delete (relative to current dialog's folder, or absolute if starts with '/')
):
    "Delete a dialog or an empty folder, stopping the dialog's kernel first if it's running"
    await stop_dialog(name)
    await _rg('DELETE', f'/api/contents/{quote(find_dname(name))}')

In [ ]:
await rm_dialog('test_dialog.ipynb')

### Frontmatter and import

Frontmatter is a fenced YAML mapping in physical message zero, whose type must be raw or note (not a prompt). Templates keep placeholder values under `formdata`. `update_frontmatter` preserves any text after the closing fence. Invalid or non-mapping headers are ordinary messages; malformed fenced YAML raises an API error.

In [ ]:
#| export
async def read_frontmatter(
    dname:str='' # Dialog to read; defaults to current dialog
)->dict|None: # The frontmatter, or None if the dialog has none
    "The YAML mapping at the start of a dialog's physical first raw or note message"
    return (await _rg('GET', '/api/frontmatter', **_target(dname)))['frontmatter']

async def update_frontmatter(
    dname:str='', # Dialog to update; defaults to current dialog
    **kwargs # Values to merge recursively into the frontmatter; `None` deletes a key
)->dict: # The updated frontmatter
    "Merge `kwargs` into a dialog's frontmatter, preserving its body or adding a raw header when absent"
    return (await _rg('PATCH', '/api/frontmatter', **_target(dname), json=kwargs))['frontmatter']

In [ ]:
tmpl = 'test_template.ipynb'
await create_or_run_dialog(tmpl)
await add_msg('Dear {{first_name}} {{last_name}}, your otter toys have shipped.', dname=tmpl)
await update_frontmatter(tmpl, type='letter', formdata=dict(first_name='John', last_name='Doe'))
fm = await update_frontmatter(tmpl, formdata=dict(last_name=None, city='Paris'))
test_eq(await read_frontmatter(tmpl), dict(type='letter', formdata=dict(first_name='John', city='Paris')))


`import_dlg` copies a template without running code, filling placeholders, refreshing the dialog or changing browser selection. It assigns fresh IDs, preserves metadata, attachments and prompt replies, and clears ordinary code outputs and execution counts. The selected physical frontmatter merges into the destination with source values winning; nested `null` deletes a key, maps merge, and arrays and scalars replace. Any nonempty frontmatter body becomes an ordinary note at the import position. A destination frontmatter header stays at message zero even when importing at the start.

Selection and copying happen in one gateway request. The default imports every unskipped message. An `ids` list imports that exact selection, including skipped messages, in source order; `ids=[]` imports nothing. Other `find_msgs` filters select the source messages in Rustygate. Frontmatter merges only when physical message zero is selected. Omitted placement follows the calling cell in the current dialog and appends to an explicitly named destination. `position='before'` without `msgid` retains the calling-cell/start convenience.


In [ ]:
#| export
@delegates(find_msgs, but=['before', 'after', 'context', 'include_output', 'include_meta', 'as_xml', 'nums', 'trunc_out', 'trunc_in', 'headers_only', 'dname'])
async def import_dlg(
    src_dname:str, # Dialog to copy messages from
    dname:str='', # Dialog to add them to; defaults to current dialog
    msgid:str=None, # Message to place them beside; default: the calling cell in its own dialog, else the end (or start for `position='before'`)
    position:str='after', # 'after' or 'before' `msgid`, or the dialog's 'start' or 'end'
    **kwargs
)->str: # The copied messages, each under a `# msg <id>` header with its new id
    "Import a clean template, merging its selected physical frontmatter into the destination"
    body = {'from': src_dname, 'msgid': msgid, 'position': position}
    if set(kwargs)=={'ids'} and isinstance(kwargs['ids'], list): body['ids'] = kwargs['ids']
    elif kwargs:
        if isinstance(kwargs.get('ids'), list): kwargs['ids'] = ','.join(kwargs['ids'])
        body['selection'] = kwargs
    res = await _rg('POST', '/api/import', **_target(dname), json=body)
    return '\n'.join(f"# msg {c['id']}\n{c['source']}" for c in res['cells'])


In [ ]:
dst = 'test_import.ipynb'
await create_or_run_dialog(dst)
await update_frontmatter(dst, type='draft', status='new')
await import_dlg(tmpl, dname=dst)
test_eq(await read_frontmatter(dst), fm | dict(status='new'))


In [ ]:
for o in (tmpl, dst): await rm_dialog(o)

In [ ]:
#| export
async def run_code_interactive(
    code:str # Code to have user run
):
    """Insert code into user's dialog and request for the user to run it. Use other functions where possible, 
    but if they can not find needed information, *ALWAYS* use this instead of guessing or giving up.
    IMPORTANT: This tool is TERMINAL - after calling it, you MUST stop all tool usage 
    and wait for user response. Never call additional tools after this one."""
    await add_msg('# Please run this:\n'+code, msg_type='code')
    return {'success': "CRITICAL: Message added to user dialog. STOP IMMEDIATELY. Do NOT call any more tools. Wait for user to run code and respond."}

`run_cells` runs code cells in a dialog's kernel through rustygate's run route: the current dialog's kernel, unless you pass another dialog's kernel id. Rustygate saves each cell's outputs in the cell and sends them to every connected client, as for a run started in Solveit, and `run_cells` returns them. aidialog's `Message.execute` and `Dialog.execute` differ: they run messages in a temporary in-process shell.

A tool can call `run_cells` while its own message is still running. Rustygate runs the requested cells in a subshell, so they don't wait behind the calling message in the main shell's queue.

When `py` runs with safepyrun's audit checks, the nested execution remains inside that audit window. `run_cells` does not add a separate sandbox.

In [ ]:
#| export
async def run_cells(
    ids:list, # Ids of code cells to run, in order
    kid:str=None, # Kernel whose dialog holds them (default: this one)
)->list: # Each cell's saved outputs
    "Run code cells in a dialog's kernel through rustygate, saving their outputs; the run stops at the first cell that raises"
    return (await _rg('POST', f"/api/kernels/{kid or dh_settings['kid']}/run", json=dict(ids=_ids(ids))))['outputs']

## Text Edit

In [ ]:
#| export
_msg_edit_doc = f"""
Be sure you've called `view_msg(…)` to ensure you know the line nums.

Message editing standard parameters:

id: Message id to edit, or list of ids, or 'all' for all messages in dialog
dname: Dialog to get info for; defaults to current dialog
out: If True, replace in output instead of content
log_changed: Add one note listing content changes in the calling dialog?

returns:
- For single id: diff of changes, or "none: No changes.", or "error: ..."
- For id list (or 'all'): list of (id,res) tuples for each message that changed or errored ("error: ..." as the res); unchanged messages are omitted"""

In [ ]:
#| export
def _msg_edit(f, name=None):
    async def wrapper(id:str|list[str], *args, out:bool=False, dname:str='', log_changed:bool=False, **kw):
        many = isinstance(id, list) or id=='all'
        ids = id if isinstance(id, list) else [id or await _caller()]
        if not ids: return []
        found = await _rg('GET', '/api/messages', **_target(dname), ids=None if id=='all' else ','.join(ids),
            context=0, include_skipped=id!='all', raw_prompt_reply=True)
        if id=='all': ids = [m['id'] for m in found]
        msgs = {m['id']:m for m in found}
        updates,logs = [],[]
        async def _one(mid):
            msg = msgs.get(_mid(mid))
            if msg is None: return f"error: No message {mid}"
            field = 'output' if out else 'content'
            text = msg.get(field, '')
            if not text: return f"error: Message has no {field}"
            try: new_text = await maybe_await(f(text, *args, **kw))
            except ValueError as e: return f'error: {e}'
            updates.append(dict(id=mid, msg={field: new_text}))
            diff = str_diff(text, new_text)
            if log_changed: logs.append(_change_log(mid, diff, dname))
            return diff or 'none: No changes.'
        results = [(mid, await _one(mid)) for mid in ids]
        await _apply_logged(dict(op='update', msgs=updates) if updates else None, logs, dname)
        return [(mid, r) for mid,r in results if not r.startswith('none:')] if many else results[0][1]
    res = splice_sig(wrapper, f, 'text')
    if name: res.__name__ = res.__qualname__ = name
    res.__doc__ = (f.__doc__ or '') + _msg_edit_doc
    return res

In [ ]:
#| export
msg_insert_line  = _msg_edit (insert_line, 'msg_insert_line')


In [ ]:
await msg_insert_line(_edit_id, 0, 'This should go to the first line')
await msg_insert_line(_edit_id, 3, 'This should go to the 4th line')
print(await msg_insert_line(_edit_id, 5, 'This should go to the last line'))

@@ -5 +5,2 @@
 This is a multiline message.
+This should go to the last line


In [ ]:
print((await read_msg(n=0, id=_edit_id, nums=True))['content'])

     1 │ This should go to the first line
     2 │ This message should be found.
     3 │ 
     4 │ This should go to the 4th line
     5 │ This is a multiline message.
     6 │ This should go to the last line


In [ ]:
#| export
msg_str_replace =  _msg_edit (str_replace, 'msg_str_replace')


In [ ]:
print(await msg_str_replace(_edit_id, 'This should go to the first line', 'This should go to the 1st line'))

@@ -1,2 +1,2 @@
-This should go to the first line
+This should go to the 1st line
 This message should be found.


In [ ]:
for f in (msg_str_replace, msg_strs_replace): assert 'use_regex' in inspect.signature(f).parameters

In [ ]:
print((await read_msg(n=0, id=_edit_id, nums=True))['content'])

     1 │ This should go to the 1st line
     2 │ This message should be found.
     3 │ 
     4 │ This should go to the 4th line
     5 │ This is a multiline message.
     6 │ This should go to the last line


In [ ]:
#| export
msg_strs_replace =  _msg_edit (strs_replace, 'msg_strs_replace')


In [ ]:
print(await msg_strs_replace(_edit_id, ['This is a multiline message.', 'This should go to the last line'], ['5th line', 'last line']))

@@ -4,3 +4,3 @@
 This should go to the 4th line
-This is a multiline message.
-This should go to the last line
+5th line
+last line


In [ ]:
print((await read_msg(n=0, id=_edit_id, nums=True))['content'])

     1 │ This should go to the 1st line
     2 │ This message should be found.
     3 │ 
     4 │ This should go to the 4th line
     5 │ 5th line
     6 │ last line


In [ ]:
#| export
msg_replace_lines =  _msg_edit (replace_lines, 'msg_replace_lines')


In [ ]:
print(await msg_replace_lines(_edit_id, 2, 4,'line 2\nline 3\nline 4\n'))

@@ -1,5 +1,5 @@
 This should go to the 1st line
-This message should be found.
-
-This should go to the 4th line
+line 2
+line 3
+line 4
 5th line


In [ ]:
print((await read_msg(n=0, id=_edit_id, nums=True))['content'])

     1 │ This should go to the 1st line
     2 │ line 2
     3 │ line 3
     4 │ line 4
     5 │ 5th line
     6 │ last line


In [ ]:
#| export
msg_del_lines =  _msg_edit (del_lines, 'msg_del_lines')


In [ ]:
print(await msg_del_lines(_edit_id, 2, 4))

@@ -1,5 +1,2 @@
 This should go to the 1st line
-line 2
-line 3
-line 4
 5th line


In [ ]:
print((await read_msg(n=0, id=_edit_id, nums=True)).content)

     1 │ This should go to the 1st line
     2 │ 5th line
     3 │ last line


In [ ]:
await del_msgs(_edit_id)

['c99d7c11']

In [ ]:
id1 = await add_msg('hello world')
id2 = await add_msg('hello there', msgid=id1)

In [ ]:
results = await msg_str_replace([id1, id2], 'hello', 'hi')
print(results)

[('b7e56484', '@@ -1 +1 @@\n-hello world\n+hi world'), ('327c264e', '@@ -1 +1 @@\n-hello there\n+hi there')]


In [ ]:
await del_msgs(id1)
await del_msgs(id2)

['327c264e']

## Prompts

`run_prompt` asks rustygate to run a prompt. With a prompt message's `msgid`, rustygate answers that prompt with the cells above it as context and saves the reply in the message, as when a user runs it in Solveit. With `text`, rustygate answers a prompt that isn't in the dialog, using the cells up to `msgid` as context, and saves nothing. Without `msgid`, that context ends at the calling cell.

The reply streams back over the comm, and `run_prompt` prints it as it arrives unless `show=False`. It returns the reply in the format Solveit stores, with each tool call as a fenced block. Interrupting the cell closes the comm, which cancels the turn, and a prompt message keeps its partial reply. A prompt run from code doesn't wait in the kernel's queue. `tools` names the tools the model may call: functions in the kernel's namespace, or rustygate's own `exec` and `spawn_agent`.

In [ ]:
#| export
async def run_prompt(
    msgid:str=None, # Prompt message to run; with `text`, the message the context ends at (default: the calling cell)
    text:str=None, # Prompt text to answer without saving anything in the dialog
    tools:list=None, # Names of the tools the model may call
    show:bool=True, # Print the reply as it streams?
    kid:str=None, # Kernel whose dialog holds the prompt (default: this one); with `text` and another dialog's kernel, pass `msgid`
)->str: # The reply, in its stored format
    "Run a prompt message, or answer `text` with the dialog as context, and return the reply"
    body = dict(cell=msgid) if text is None else dict(text=text, msgid=msgid or await _caller())
    if tools: body['tools'] = tools
    async with http_client(**_http_options()) as c, c.stream('POST', f"/api/kernels/{kid or dh_settings['kid']}/prompt", json=body) as r:
        r.raise_for_status()
        async for line in r.aiter_lines():
            d = json.loads(line)
            if 'reply' in d: return d['reply']
            if show: print(d['text'], end='', flush=True)

In [ ]:
#| eval: false
pid = await add_msg('Reply with just the word: hello', msg_type='prompt')
reply = await run_prompt(pid)

In [ ]:
#| eval: false
test_eq((await read_msgid(pid))['output'], reply)
await del_msgs(pid)

['a3c96594']

In [ ]:
#| eval: false
await run_prompt(text='What is 2+2? Reply with just the number.')

'Whoops! An error (error sending request for url (http://127.0.0.1:9/models)) occurred while processing your request.\nIf this problem persists, please contact us on Discord.\nPlease include your dialog url and error info in your message.'

## Help

In [ ]:
#| export
async def solveit_docs():
    """Full reference documentation for Solveit - use this to answer questions about how to use Solveit.
    **NB**: The whole docs fit in LLM context, so read the whole thing, don't search/filter it. *Always* re-run rather than relying on truncated history or assumptions."""
    from ghapi.core import GhApi
    _ref_gist_id = '9e7b444aba5ecf6d14295ba2cee890c3'
    pre = f"""⚠️ This content will be truncated in your next turn. Re-call this function if you need it again.
If the user wants more info, give them a link to https://gist.github.com/jph00/{_ref_gist_id}."""
    return pre + (await GhApi().gist_file(_ref_gist_id))['content']

In [ ]:
#| export
def dialog_link(
    dname:str='', # Dialog to link to (relative to current dialog's folder, or absolute if starts with '/'); empty for in-page anchor link
    msg_id:str=None # Optional message id to scroll to
):
    """Return an IPython HTML link to open a dialog in Solveit.
    After calling this, output the resulting HTML anchor tag exactly as returned—do not wrap in a fenced code block or convert to markdown link format."""
    if not (dname or msg_id): return 'err: no dname or msg_id'
    url = ''
    if dname:
        dname = find_dname(dname)
        url += f"/dialog_?{urlencode({'name': dname})}"
    if msg_id: url += '#_'+_mid(msg_id)  # the fragment targets the DOM id, which carries a `_` prefix
    return HTML(f'<a href="{url}" target="_blank">{dname}</a>') if dname else Markdown(f'[{url}]({url})')


In [ ]:
dialog_link(msg_id='a7d82acd')

[#_a7d82acd](#_a7d82acd)

In [ ]:
dialog_link(dname='/CRAFT.ipynb')

HTML(<a href="/dialog_?name=CRAFT.ipynb" target="_blank">CRAFT.ipynb</a>)

In [ ]:
dialog_link(dname='/CRAFT.ipynb', msg_id='ce727fd8')

HTML(<a href="/dialog_?name=CRAFT.ipynb#_ce727fd8" target="_blank">CRAFT.ipynb</a>)

## export -

In [ ]:
#| hide
from nbdev import nbdev_export
nbdev_export()

testing